In [ ]:
%load_ext autoreload
%autoreload 2

# Публичный API `ifc_checker_script`

Этот notebook показывает все публичные функции пакета `ifc_checker_script` и примеры их использования без графического интерфейса.

In [ ]:
from pathlib import Path
import sys
sys.path.append(r'../')
from ifc_checker_script import (
    check_ifc,
    delete_skipped,
    delete_skipped_from_one_html,
    ifc_exist_in_ids_dict,
    read_json_config,
    save_json_config_template,
)



repo_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
samples_dir = repo_root / 'samples'
output_dir = repo_root / '.output' / 'notebook_api'
output_dir.mkdir(parents=True, exist_ok=True)

repo_root

## `read_json_config(config_path)`

Читает JSON-конфигурацию и возвращает словарь сопоставления между именами IDS и шаблонами IFC-файлов.

In [ ]:
config_path = samples_dir / 'sample_config.json'
configuration = read_json_config(str(config_path))
configuration

## `ifc_exist_in_ids_dict(ids_file, ifc_file, json_config)`

Проверяет, должен ли IFC-файл участвовать в проверке по конкретному IDS на основе словаря конфигурации.

In [ ]:
ifc_exist_in_ids_dict('test_AR', 'Здание_1_AR', configuration)

## `save_json_config_template(path_to_save)`

Сохраняет шаблон JSON-конфигурации для дальнейшего ручного редактирования.

In [ ]:
template_base_path = output_dir / 'sample_template'
save_json_config_template(str(template_base_path))
template_base_path.with_suffix('.json')

## `delete_skipped_from_one_html(html_path)`

Удаляет из одного HTML-отчета секции, содержащие статус `item skipped`.

In [ ]:
single_html_path = output_dir / 'single_skipped_example.html'
single_html_path.write_text(
    '''
    <html><body>
        <section><h2>Skipped</h2><span class="item skipped">Skipped</span></section>
        <section><h2>Passed</h2><span class="item passed">Passed</span></section>
    </body></html>
    ''',
    encoding='utf-8',
)

delete_skipped_from_one_html(str(single_html_path))

## `delete_skipped(folder_path)`

Проходит по папке с HTML-отчетами и очищает все файлы от секций со статусом `skipped`.

In [ ]:
delete_skipped(str(output_dir))

## `check_ifc(folder_path_ifc, files_path_ids, folder_path_report, configuration=None, status_callback=None)`

Запускает полную проверку IFC-моделей по IDS-файлам, формирует отдельные HTML-отчеты, сводный отчет и лог.

Ниже показан пример запуска на sample-данных проекта.

In [ ]:
ids_files = [
    samples_dir / 'test_all.ids',
    samples_dir / 'test_AR.ids',
    samples_dir / 'test_KR.ids',
    samples_dir / 'test_IOS.ids',
]

report_dir = output_dir / 'reports'
messages = []

result = check_ifc(
    folder_path_ifc=str(samples_dir / 'ifc_models'),
    files_path_ids=[str(path) for path in ids_files],
    folder_path_report=str(report_dir),
    configuration=configuration,
    status_callback=messages.append,
    delete_empty_checks=False,
)

result

## Сообщения о ходе проверки

Если передать `status_callback`, функция будет отдавать в него текстовые статусы выполнения.

In [ ]:
messages